当准备轴承故障信号数据时，需要考虑两个方面：
1. 轴承信号数据和对应的标签。
2. 同时，为了构建图结构，还需要确定节点之间的连接关系。

In [1]:
from joblib import dump, load

# 加载数据
train_set = load('./dataresult/train_set') 
val_set = load('./dataresult/val_set') 
test_set = load('./dataresult/test_set') 
print(train_set.shape)  # 二维数组，行代表样本数量， 列代表信号长度和标签
print(val_set.shape)
print(test_set.shape)

(1631, 1025)
(466, 1025)
(233, 1025)


# 创建图数据

使用torch.tensor将轴承信号数据、边的连接关系和标签转换为张量形式。具体地，使用torch.tensor函数将信号数据转换为torch.float类型的张量，边的连接关系转换为torch.long类型的张量，标签转换为torch.long类型的张量。这样就可以创建一个Data对象，其中x属性表示节点特征（轴承信号数据），edge_index属性表示边的连接关系，y属性表示节点的标签

In [ ]:
from joblib import dump, load
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from torch_geometric.data import Data

import pandas as pd
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader 
from sklearn.metrics import accuracy_score
from torch_geometric.nn import global_mean_pool  # 导入全局池化层

# ---------------------- 定义GNN模型 ----------------------
class GNNModel(torch.nn.Module):
    def __init__(self, input_dim=33, hidden_dim=16, output_dim=2):  # 默认输入维度调整为
        super(GNNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)  # 第二层保持隐藏层维度
        self.fc = torch.nn.Linear(hidden_dim, output_dim)  # 全连接层

    def forward(self, x, edge_index, edge_weight, batch):
        # 1. 节点特征提取
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        x = F.relu(x)
        
        # 2. 全局池化：将节点特征转换为图级特征
        x = global_mean_pool(x, batch)  # batch 参数指示节点属于哪个图
        
        # 3. 全连接层输出分类结果
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

# ---------------------- 训练和评估函数 ----------------------
def train(model, data_loader, optimizer):
    model.train()
    total_loss = 0
    for data in data_loader:
        optimizer.zero_grad()
        out = model(
            data.x, 
            data.edge_index, 
            data.edge_attr.view(-1),  # 展平边权重
            data.batch  # 新增 batch 参数
        )
        loss = F.nll_loss(out, data.y)  # data.y 是图级标签
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

def evaluate(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for data in data_loader:
            out = model(
                data.x, 
                data.edge_index, 
                data.edge_attr.view(-1),
                data.batch
            )
            pred = out.argmax(dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

# -------------------------------------------
# 1. 排除自环边: 在构建边时过滤i==j的情况
# 2. 数据标准化: 对信号值和时间步分别进行标准化
# 3. 时间步特征增强: 将时间步作为第二维特征
# 4. 边权重保留: 将KNN距离作为边属性存储
# 5. K值交叉验证: 新增交叉验证函数选择最佳K值
# -------------------------------------------

def make_NearestNeighbors(signals, m_k=5):
    """改进后的KNN函数，返回节点距离和索引"""
    nn = NearestNeighbors(n_neighbors=m_k+1, metric='euclidean')  # 多找一个邻居用于排除自环
    nn.fit(signals)
    distances, indices = nn.kneighbors(signals)
    return distances[:, 1:], indices[:, 1:]  # 排除第一个自环邻居

def get_edge_indexs(signals, indices, distances):
    """改进后的边构建函数，添加边权重并排除自环边"""
    edge_index = []
    edge_weights = []
    eps = 1e-5
    for i in range(len(signals)):
        # 验证当前节点的邻居数据
        assert len(indices[i]) == len(distances[i]), f"节点{i}的邻居索引与距离不匹配"
        
        for j_idx, j in enumerate(indices[i]):
            # 验证索引有效性
            if j >= len(signals):
                raise ValueError(f"节点{i}的邻居索引{j}超出范围")
                
            if i != j:
                dist = distances[i][j_idx]
                # 验证距离值有效性
                if dist is None or np.isnan(dist):
                    raise ValueError(f"节点{i}到{j}的距离无效: {dist}")
                
                weight = 1.0 / (dist + eps)
                edge_index.append([i, j])
                edge_weights.append(weight)
    edge_index = np.array(edge_index).T
    edge_weights = np.array(edge_weights)
    
    # 转换为tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_weights = torch.tensor(edge_weights, dtype=torch.float).view(-1, 1)
    return edge_index, edge_weights

# ---------------------- 新增正弦编码函数 ----------------------
def sinusoidal_encode(position, d_model=32):
     """Transformer风格的正弦位置编码"""
     position = position.astype(float)
     div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
     encoding = np.zeros((len(position), d_model))
     for i in range(len(position)):
         pos = position[i][0]  # 从二维数组中提取标量值
         encoding[i, 0::2] = np.sin(pos * div_term)
         encoding[i, 1::2] = np.cos(pos * div_term)
     return encoding

def temporal_feature_augmentation(signal):
    """时间步特征增强：添加标准化后的时间步作为第二维特征"""
    # 信号标准化
    signal_scaler = StandardScaler()
    scaled_signal = signal_scaler.fit_transform(signal)
    
# 生成正弦位置编码
    time_positions = np.arange(len(signal)).reshape(-1, 1)
    time_encoding = sinusoidal_encode(time_positions)  # 

    # 合并特征
    return np.hstack((scaled_signal, time_encoding))

def make_graph_dataset(dataframe, m_k):
    """改进后的图数据生成函数，整合所有新功能"""
    x_data = dataframe.iloc[:,0:-1].values
    y_labels = dataframe.iloc[:,-1].values

    dataset = []  
    for index in range(dataframe.shape[0]):
        # 原始信号处理
        raw_signal = x_data[index].reshape(-1, 1)
        
        # 特征增强：信号标准化 + 时间步特征
        #features = temporal_feature_augmentation(raw_signal)
        features = temporal_feature_augmentation(raw_signal)  
        
        # 构建KNN关系
        distances, indices = make_NearestNeighbors(features, m_k=m_k)
        
        # 构建边和边权重
        edge_index, edge_attr = get_edge_indexs(features, indices, distances)
        
        # 创建Data对象
        data = Data(
            x=torch.tensor(features, dtype=torch.float),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(y_labels[index], dtype=torch.long)
        )
        dataset.append(data)
    
    return dataset

def cross_validate_k(train_set, k_candidates):
    """K值交叉验证函数"""
    best_k = k_candidates[0]
    best_score = -np.inf
    
    for k in k_candidates:
        print(f"\n--- 正在验证K={k} ---")
        fold_scores = []
        
        # 创建K折交叉验证
        kf = KFold(n_splits=5)
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_set)):
            # 生成训练/验证子集
            train_sub = train_set.iloc[train_idx]
            val_sub = train_set.iloc[val_idx]
            
            # 生成图数据
            train_graph = make_graph_dataset(train_sub, k)
            val_graph = make_graph_dataset(val_sub, k)
            
            # 数据加载器
            train_loader = DataLoader(train_graph, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_graph, batch_size=32, shuffle=False)
            
            # 初始化模型、优化器
            input_dim = train_graph[0].x.shape[1]  # 输入特征维度
            output_dim = len(np.unique(train_set.iloc[:,-1]))  # 输出类别数
            model = GNNModel(input_dim=input_dim, hidden_dim=16, output_dim=output_dim)
            #model = GNNModel(hidden_dim=16, output_dim=output_dim)  # 输入维度由特征自动决定
            optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
            
            # 训练模型
            for epoch in range(10):  # 简单训练10个epoch
                train(model, train_loader, optimizer)
            
            # 评估模型
            score = evaluate(model, val_loader)
            fold_scores.append(score)
            print(f"Fold {fold+1}: 验证集准确率 = {score:.4f}")
        
        avg_score = np.mean(fold_scores)
        print(f"K={k}, 平均验证集准确率 = {avg_score:.4f}")
        if avg_score > best_score:
            best_score = avg_score
            best_k = k
    
    print(f"\n最佳K值: {best_k} (得分: {best_score:.4f})")
    return best_k

# ---------------------- 主程序 ----------------------
if __name__ == "__main__":

    # 步骤1：通过交叉验证选择最佳K值（需要实现模型部分）
    k_candidates = [3,5,7,11,13]
    best_k = cross_validate_k(train_set, k_candidates)
    #best_k = 5  # 假设交叉验证选择的结果
    
    # 步骤2：使用最佳K值生成最终数据集
    print(f"\n使用最佳K值 {best_k} 生成图数据集...")
    train_graph_data = make_graph_dataset(train_set, best_k)
    val_graph_data = make_graph_dataset(val_set, best_k)
    test_graph_data = make_graph_dataset(test_set, best_k)

    # 保存数据
    dump(train_graph_data, './dataresult/train_graph_data')
    dump(val_graph_data, './dataresult/val_graph_data')
    dump(test_graph_data, './dataresult/test_graph_data')
    
    print("图数据保存完成！")


--- 正在验证K=3 ---
Fold 1: 验证集准确率 = 0.5780
Fold 2: 验证集准确率 = 0.6012
Fold 3: 验证集准确率 = 0.6074
Fold 4: 验证集准确率 = 0.5276
Fold 5: 验证集准确率 = 0.4479
K=3, 平均验证集准确率 = 0.5524

--- 正在验证K=5 ---
Fold 1: 验证集准确率 = 0.6911
Fold 2: 验证集准确率 = 0.6902
Fold 3: 验证集准确率 = 0.6411
Fold 4: 验证集准确率 = 0.6810
Fold 5: 验证集准确率 = 0.4724
K=5, 平均验证集准确率 = 0.6352

--- 正在验证K=7 ---
Fold 1: 验证集准确率 = 0.7034
Fold 2: 验证集准确率 = 0.6779
Fold 3: 验证集准确率 = 0.6840
Fold 4: 验证集准确率 = 0.7393
Fold 5: 验证集准确率 = 0.5307
K=7, 平均验证集准确率 = 0.6671

--- 正在验证K=11 ---
Fold 1: 验证集准确率 = 0.5902
Fold 2: 验证集准确率 = 0.5890
Fold 3: 验证集准确率 = 0.6503
Fold 4: 验证集准确率 = 0.7362
Fold 5: 验证集准确率 = 0.7117
K=11, 平均验证集准确率 = 0.6555

--- 正在验证K=13 ---
Fold 1: 验证集准确率 = 0.6330
Fold 2: 验证集准确率 = 0.6687
Fold 3: 验证集准确率 = 0.7086
Fold 4: 验证集准确率 = 0.7055
Fold 5: 验证集准确率 = 0.6442
K=13, 平均验证集准确率 = 0.6720

最佳K值: 13 (得分: 0.6720)

使用最佳K值 13 生成图数据集...
图数据保存完成！


In [3]:
# 看一个 数据集结构
test_graph_data  

[Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=1),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=3),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=9),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=0),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=7),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=2),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=1),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=8),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=1),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=8),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=6),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=8),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=0),
 Data(x=[1024, 33], edge_index=[2, 13312], edge_attr=[13312, 1], y=0),
 Data(